# Training Loop

In [ ]:
# Cell 1: Setup autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# Setup Experiment and run training
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
import copy

paths = get_project_paths()

import logging

# Set root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for your notebook/script
logger = logging.getLogger(__name__)

In [ ]:
paths = get_project_paths()
huxt_run_id = 3

config = TrainingConfig(
    huxt_run_id=huxt_run_id,
    huxt_data_path=paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'discontinuities.npy',
    
    model_type = 'ensemble'
    n_ensembles=100,

    ensemble_regressors={
        'LinearRegression': LinearRegression,
        # 'Ridge': lambda: Ridge(alpha=1.0, positive=True),
        # 'DecisionTree': lambda: DecisionTreeRegressor(max_depth=10, random_state=42),
        # 'RandomForest': lambda: RandomForestRegressor(n_estimators=30, max_depth=10),
        # 'XGBoost_Fast': lambda: XGBRegressor(
        #     n_estimators=100,      # Fewer trees (faster)
        #     max_depth=5,           # Shallow trees (faster, less overfit)
        #     learning_rate=0.05,    # Higher LR with fewer trees
        #     subsample=0.8,
        #     colsample_bytree=0.8,
        #     tree_method='hist',    # Histogram-based (much faster!)
        #     random_state=42,
        #     n_jobs=1
        # )
    },
    
    # Update constraints for each model
    model_constraints={
        'LinearRegression': 'realistic',  # Needs constraints
        'Ridge': None,                    # Has positive=True
        'DecisionTree': None,
        'RandomForest': 'realistic',      # May predict negatives
        'XGBoost_Fast': None,                  
    },

    lead_times=[24],
    storm_thresholds=[4.5,],
    random_seeds=[42],
    test_folds=[0],
    omni_subsets=[["Bz_GSM"]],
    balance=False,
    save_results=True,
    n_jobs=-1,  # Use all CPUs
)

# Run training
run_training_pipeline(config)

## MLP

### Phase 1 - Features

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Experiment 001: Baseline - V only
# Phase 1: Feature exploration
config_phase1 = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_use_ensemble_statistics=True,
    mlp_n_epochs=100,
    mlp_learning_rate=0.001,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    # Test all 10 feature combinations in one run
    omni_subsets=[
        [],                           # 1. Baseline: V only
        ['Bz_GSM'],                   # 2. IMF Bz (most important for storms)
        ['Dst'],                      # 3. Ring current index
        ['n_sw'],                     # 4. Density
        ['flow_pressure'],            # 5. Dynamic pressure
        ['Bz_GSM', 'Dst'],           # 6. Best two combined
        ['Bz_GSM', 'n_sw'],          # 7. IMF + density
        ['Bz_GSM', 'flow_pressure'], # 8. IMF + pressure
        ['Bz_GSM', 'Dst', 'n_sw'],   # 9. Top three
        ['Bz_GSM', 'Dst', 'flow_pressure', 'E_field'],  # 10. Kitchen sink
    ],
    
    experiment_phase='phase1_features',
)

run_training_pipeline(config_phase1)

### Phase 1 - Ensemble Spread

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase1_ensemble_spread = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    # mlp_ensemble_percentiles=[50], # don't alter this param within experiment loop
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
mlp_ensemble_percentiles = [
    # Basic configurations
    [50],                           # 1. Median only (minimal)
    [5, 50, 95],                    # 2. Baseline: median + 90% bounds
    [25, 50, 75],                   # 3. Interquartile range
    
    # More detailed distributions
    [5, 25, 50, 75, 95],            # 4. Quintiles (5 points)
    [10, 30, 50, 70, 90],           # 5. Alternative quintiles
    [5, 16, 50, 84, 95],            # 6. Roughly ±1σ, ±2σ for normal
    
    # Very detailed
    [5, 16, 33, 50, 67, 84, 95],    # 7. Seven percentiles
    [1, 10, 25, 50, 75, 90, 99],    # 8. Extreme tails included
    
    # Asymmetric (more detail in tails)
    [1, 5, 25, 50, 75, 95, 99],     # 9. Focus on extremes
    [5, 10, 25, 50, 95],            # 10. Asymmetric (sparse upper tail)
    ]

for ens_p in mlp_ensemble_percentiles: 
    config_phase1_ensemble_spread.mlp_ensemble_percentiles = ens_p
    run_training_pipeline(config_phase1_ensemble_spread)

### Phase 2 - Architecture taper

In [ ]:
import logging

# Set root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for your notebook/script
logger = logging.getLogger(__name__)

# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase2_architecture_taper = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    mlp_ensemble_percentiles=[5, 50, 95, ],
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
# Phase 2: Architecture experiments
architecture_configs = [
    # Small but capable
    ([300, 150, 75], 'baseline'),       # ~565k params
    ([256, 128, 64], 'taper_pow2'),     # ~242k params
    ([200, 100, 50], 'taper_small'),    # ~195k params
    ([150, 75, 40], 'taper_tiny'),      # ~135k params
    
    # Uniform (more regularization from structure)
    ([200, 200, 200], 'uniform_200'),   # ~234k params
    ([150, 150, 150], 'uniform_150'),   # ~155k params
    
    # Depth tests (keeping narrow)
    ([200, 100], 'depth_2'),            # ~193k params
    ([200, 150, 100, 50], 'depth_4'),   # ~228k params
    
    # Extreme compression (risky but interesting)
    ([100, 100, 100], 'uniform_100'),   # ~107k params
    ([400, 50, 400], 'bottleneck_extreme'), # ~373k params - force feature learning
]

# Base config with BEST settings from Phase 1
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=100, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.001,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase2_architecture',
)


for architecture, arch_name in architecture_configs:
    logger.info(f"\n{'='*80}")
    logger.info(f"Testing architecture: {arch_name}")
    logger.info(f"{'='*80}\n")
    
    config = copy.deepcopy(base_config)
    config.mlp_architecture = architecture
    config.model_name = f"MLP_{arch_name}"  # Optional: override base name
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 2 architecture experiments complete!")

## Phase 2 - Unbalanced training and testing set 

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase2_architecture_taper = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    mlp_ensemble_percentiles=[5, 50, 95, ],
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
# Phase 2: Architecture experiments
architecture_configs = [
    # Small but capable
    ([300, 150, 75], 'baseline'),       # ~565k params
    ([256, 128, 64], 'taper_pow2'),     # ~242k params
    ([200, 100, 50], 'taper_small'),    # ~195k params
    ([150, 75, 40], 'taper_tiny'),      # ~135k params
    
    # Uniform (more regularization from structure)
    ([200, 200, 200], 'uniform_200'),   # ~234k params
    ([150, 150, 150], 'uniform_150'),   # ~155k params
    
    # Depth tests (keeping narrow)
    ([200, 100], 'depth_2'),            # ~193k params
    ([200, 150, 100, 50], 'depth_4'),   # ~228k params
    
    # Extreme compression (risky but interesting)
    ([100, 100, 100], 'uniform_100'),   # ~107k params
    ([400, 50, 400], 'bottleneck_extreme'), # ~373k params - force feature learning
]

# Base config with BEST settings from Phase 1
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase2_architecture_unbalanced_storms',
)


for architecture, arch_name in architecture_configs:
    logger.info(f"\n{'='*80}")
    logger.info(f"Testing architecture: {arch_name}")
    logger.info(f"{'='*80}\n")
    
    config = copy.deepcopy(base_config)
    config.mlp_architecture = architecture
    config.model_name = f"MLP_{arch_name}"  # Optional: override base name
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 2 architecture experiments complete!")

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig

base_loss_config = NLLLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],

    loss_config = base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_nll',
)

loss_configs = (
    (1, 1),
    (1, 0.5),
    (0.5, 1),
    (1, 0.33),
    (0.33, 1),
)

for w_sigma, w_accuracy in loss_configs:
    config = copy.deepcopy(base_config)
    
    config.loss_config.w_sigma = w_sigma
    config.loss_config.w_accuracy = w_accuracy
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 3 Loss Function experiments complete!")

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig
paths = get_project_paths()

base_loss_config = NLLLossConfig()
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_hp30_weighting',
)

intensity_types = [None, 'linear', 'quadratic', 'threshold']

for intensity_type in intensity_types:
    config = copy.deepcopy(base_config)
    config.loss_config.intensity_type = intensity_type
    config.experiment_phase = f"phase3_loss_functions_hp30_weighting"
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed intensity_type={intensity_type}: {e}")
        continue

logger.info("\nPhase 3 Hp30 weighting experiments complete!")

## Phase 3 Loss Functions - Hp30 weighting

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig
paths = get_project_paths()

base_loss_config = NLLLossConfig()
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / f'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',

    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=True,  
    
    mlp_n_epochs=1000,
    mlp_learning_rate=0.01,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    balance=False,
    save_results=True,
    remove_cmes=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    
    experiment_phase='phase3_loss_functions_hp30_weighting',
)

intensity_configs = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]

for scale_factor in intensity_configs:
    config = copy.deepcopy(base_config)
    config.loss_config.intensity_type = 'linear' if scale_factor is not None else None
    config.loss_config.intensity_strength = scale_factor if scale_factor is not None else 1.0
    config.experiment_phase = 'phase3_loss_functions_hp30_weighting_scale_linear'

    label = f"linear scale={scale_factor}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("\nPhase 3 Hp30 weighting experiments complete!")

In [ ]:
## Phase 3 - Loss Functions CRPS Loss 

In [ ]:
from storm_regression.training_pipeline import NLLLossConfig, CRPSLossConfig, AsymmetricLossConfig
paths = get_project_paths()

# Configure CRPS Loss
crps_loss_config = CRPSLossConfig()

# Configure NLL Loss
nll_loss_config = NLLLossConfig()
nll_loss_config.intensity_type = 'linear' 
nll_loss_config.intensity_strength = 2.0

# Configure Asymmetric Loss
asym_loss_config = AsymmetricLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,

    mlp_n_epochs=1000,
    mlp_learning_rate=0.01,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],

    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    balance=False,
    save_results=True,
    remove_cmes=True,

    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_variety',
)

try:
    for loss_config in [crps_loss_config, nll_loss_config, asym_loss_config]:
        config = copy.deepcopy(base_config)
        config.loss_config = loss_config
        logger.info(f"Running: {config.loss_config.loss_type} loss")
        run_training_pipeline(config)
except Exception as e:
    logger.error(f"Failed {config.loss_config.loss_type} loss: {e}")

logger.info("Phase 3 Loss Function Variety experiment complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()

crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.010,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_selection',
)

# Each entry is:
# (filter_ensemble, n_ensemble_keep, ensemble_selection_method, mlp_ensemble_percentiles)
ensemble_configs = [
    # ── Baseline: no filtering, snap ─────────────────────────────────────
    # (False, 200, 'snap',         [5, 16, 50, 84, 95]),

    # ── Filtering with snap, variety of n_keep ───────────────────────────
    (True,  100, 'snap',         [5, 16, 50, 84, 95]),
    (True,   50, 'snap',         [5, 16, 50, 84, 95]),
    (True,   25, 'snap',         [5, 16, 50, 84, 95]),
    (True,   10, 'snap',         [5, 16, 50, 84, 95]),

    # ── Filtering with per_timestep, variety of n_keep ───────────────────
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   25, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   10, 'per_timestep', [5, 16, 50, 84, 95]),

    # ── No filtering, per_timestep ────────────────────────────────────────
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),

    # ── Wider percentile range ────────────────────────────────────────────
    (True,   50, 'snap',         [1, 5, 16, 50, 84, 95, 99]),
    (True,   50, 'per_timestep', [1, 5, 16, 50, 84, 95, 99]),

    # ── Narrower percentile range ─────────────────────────────────────────
    (True,   50, 'snap',         [25, 50, 75]),
    (True,   50, 'per_timestep', [25, 50, 75]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble             = filter_ensemble
    config.n_ensemble_keep             = n_keep
    config.ensemble_selection_method   = selection_method
    config.mlp_ensemble_percentiles    = percentiles

    label = (f"filter={filter_ensemble}, n_keep={n_keep}, "
             f"method={selection_method}, percentiles={percentiles}")

    perc_str = '-'.join(str(p) for p in percentiles)
    config.run_name = (
        f"filter{filter_ensemble}"
        f"_keep{n_keep}"
        f"_{selection_method}"
        f"_p{perc_str}"
    )

    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble selection experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_with_raw_ensembles',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,    5, 'snap', [5, 16, 50, 84, 95]),
    (True,   10, 'snap', [5, 16, 50, 84, 95]),
    (True,   20, 'snap', [5, 16, 50, 84, 95]),
    (True,   25, 'snap', [5, 16, 50, 84, 95]),
    (True,   50, 'snap', [5, 16, 50, 84, 95]),
    (True,   75, 'snap', [5, 16, 50, 84, 95]),
    (True,  100, 'snap', [5, 16, 50, 84, 95]),
    (True,  150, 'snap', [5, 16, 50, 84, 95]),
    (False, 200, 'snap', [5, 16, 50, 84, 95]),  # full ensemble baseline

    # ── n_keep sweep, denser percentiles ─────────────────────────────────
    (True,   10, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,   25, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,   50, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,  100, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (False, 200, 'snap', [5, 10, 25, 50, 75, 90, 95]),

    # ── n_keep sweep, very dense percentiles ─────────────────────────────
    (True,   50, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (True,  100, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (False, 200, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),

    # ── Just median (minimal features) ───────────────────────────────────
    (True,   50, 'snap', [50]),
    (False, 200, 'snap', [50]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_per_timestep',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,    5, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   10, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   20, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   25, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   75, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  150, 'per_timestep', [5, 16, 50, 84, 95]),
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),  # full ensemble baseline

    # ── n_keep sweep, denser percentiles ─────────────────────────────────
    (True,   10, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,   25, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,   50, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,  100, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (False, 200, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),

    # ── n_keep sweep, very dense percentiles ─────────────────────────────
    (True,   50, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (True,  100, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (False, 200, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),

    # ── Just median (minimal features) ───────────────────────────────────
    (True,   50, 'per_timestep', [50]),
    (False, 200, 'per_timestep', [50]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_per_timestep_crps_loss',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,   20, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  150, 'per_timestep', [5, 16, 50, 84, 95]),
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),  # full ensemble baseline

]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

## Phase 4: twCRPS

### Formulation testing

In [ ]:
from storm_regression.training_pipeline import tw_crps_lognormal_loss, crps_lognormal_loss
import torch

mu = torch.tensor([1.0, 1.5])
sigma = torch.tensor([0.4, 0.3])
y = torch.tensor([3.0, 5.0])

# (a) threshold below all data -> v is linear -> twCRPS == ordinary CRPS
torch.manual_seed(0); a = tw_crps_lognormal_loss(mu, sigma, y, threshold=-10, sharpness=1.0, n_samples=50000)
torch.manual_seed(0); b = crps_lognormal_loss(mu, sigma, y, n_samples=50000)   # your existing

passed = (float(a) - float(b)) < 1e-3
print(f"Test 1: {'passed' if passed else 'failed'}")
print(float(a), float(b), "\n")   # should match to ~1e-3

# (b) gradient flows
mu_ = mu.clone().requires_grad_(True)
tw_crps_lognormal_loss(mu_, sigma, y, threshold=4.66).backward()
print(f"Test 2: passed if tensor is finite and non-zero")
print(mu_.grad)             # finite, non-zero


In [ ]:
from storm_regression.training_pipeline import run_training_pipeline
from storm_utils.config_paths import get_project_paths

import logging
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, NLLLossConfig, CRPSLossConfig, TwCRPSLossConfig
)

logger = logging.getLogger(__name__)
paths = get_project_paths()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),                 # overwritten below
)

# ONLY the loss varies. Intensity weighting OFF on nll/crps (the improper path twCRPS replaces).
loss_configs = [
    # ('nll',          NLLLossConfig(intensity_strength=0)),
    ('crps',         CRPSLossConfig(intensity_strength=0)),
    ('twcrps_t4.66', TwCRPSLossConfig(threshold=4.66, sharpness=2.0, n_samples=3000)),
    ('twcrps_t5.0',  TwCRPSLossConfig(threshold=5.0,  sharpness=2.0, n_samples=3000)),
    ('twcrps_t6.5',  TwCRPSLossConfig(threshold=6.5,  sharpness=2.0, n_samples=3000)),
]

for tag, lc in loss_configs:
    config = copy.deepcopy(base_config)
    config.loss_config = lc
    config.experiment_phase = f'phase4_loss_comparison'
    config.run_name = f'loss_{tag}'
    try:
        logger.info(f"Running loss = {tag}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed loss={tag}: {e}")
        continue

logger.info("Phase 4 twCRPS loss comparison complete!")

In [ ]:
## Mixture Overfit Test

import torch
from torch.utils.data import Subset, DataLoader
from storm_utils.data_structure import ForecastingDataset
from storm_utils.config_paths import get_project_paths
from storm_regression.training_pipeline import train_mlp, MixtureNLLLossConfig

paths = get_project_paths()

# 1. Build dataset exactly as the pipeline does
dataset = ForecastingDataset(
    parquet_path=str(paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet'),
    discontinuity_path=str(paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy'),
    seed=42,
    Nens=200,
    lead_time_hours=12,
    forecast_duration_hours=24,
    stride_hours=24,
)

# 2. Match the pipeline's preprocessing (no balance, remove CMEs, set OMNI)
dataset.remove_cmes(inplace=True)
train_indices, test_indices = dataset.rotation_aligned_train_test_split(
    train_ratio=0.8, test_fold=0
)
dataset.set_omni_columns(['Bz_GSM', 'Dst', 'n_sw'])

# 3. Grab a TINY slice of the training set — deliberately include storms
#    (pick indices spread across the set so it's not all-quiet)
tiny_indices = train_indices[:50]                     # first 50 train windows
tiny = Subset(dataset, tiny_indices)
tiny_loader = DataLoader(tiny, batch_size=50, shuffle=True)

# 4. Overfit: high patience + many epochs so it can memorise
model, scaler = train_mlp(
    tiny_loader, n_ensembles=200,
    loss_config=MixtureNLLLossConfig(n_components=2),
    include_ensemble_spread=True,
    ensemble_percentiles=[5,16,50,84,95],
    ensemble_selection_method='snap', filter_ensemble=False,
    architecture=[150,150,150],
    n_epochs=500,
    learning_rate=0.0005,      # 10x lower — the key change
    patience=500,
    device='cpu', debug=False,
)

In [ ]:
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.001,        # lower than single-LogNormal's 0.005 — mixtures are LR-sensitive
    mlp_patience=20,                 # real early stopping (NOT the disabled patience from the overfit test)
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],               # single run first to confirm it works end-to-end
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase='phase5_mixture_nll',
    run_name='mixture_K2_nll_highLR',
)

run_training_pipeline(base_config)

In [ ]:
# Confirmation that the Mixture is being trained properly

from pathlib import Path
from storm_regression.case_study_analysis import load_results
from storm_regression.predictive import forecast_from_results

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_head_sanity_check'
results_path = results_dir / 'results_loss_mixture_nll.pkl'

results, config, _ = load_results(results_path)

mu  = np.asarray(results['mu_m'])      # (B, K)
sig = np.asarray(results['sigma_m'])   # (B, K)
w   = np.asarray(results['alpha'])     # (B, K)

# 1. Are the two components' means actually separated, per window?
mu_gap = np.abs(mu[:, 0] - mu[:, 1])             # (B,)
print(f"|mu_0 - mu_1|:   mean={mu_gap.mean():.3f}  median={np.median(mu_gap):.3f}  max={mu_gap.max():.3f}")

# 2. Are both components ever actually used? (weights not pinned to one)
print(f"weight on comp 0: mean={w[:,0].mean():.3f}  min={w[:,0].min():.3f}  max={w[:,0].max():.3f}")
frac_decisive = np.mean((w.max(axis=1) > 0.95))  # fraction of windows dominated by one comp
print(f"fraction of windows >95% one component: {frac_decisive:.2%}")

# 3. Do the components separate MORE on storm windows? (the thing we WANT)
storm = results['y_test'] > 4.66
print(f"mean |mu gap| quiet={mu_gap[~storm].mean():.3f}  storm={mu_gap[storm].mean():.3f}")

# 4. Verdict
if mu_gap.mean() < 0.05 and w[:,0].std() < 0.05:
    print(">>> LIKELY COLLAPSED: components are nearly identical and weights barely vary.")
else:
    print(">>> Components are distinct and/or weights adapt per window — not collapsed.")

## A quick LR and Patience test 

In [ ]:
import copy
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase='phase5_mixture_nll',
)

# (learning_rate, patience) combinations
grid = [
    (0.0001, 20),
    (0.001,  20),
    (0.0001, 60),
    (0.001,  60),
]

for lr, patience in grid:
    config = copy.deepcopy(base_config)
    config.mlp_learning_rate = lr
    config.mlp_patience      = patience
    lr_tag = f"lr{lr:g}".replace('.', 'p')        # e.g. lr0p001
    pat_tag = f"pat{patience}"
    config.run_name = f"mixture_K2_nll_{lr_tag}_{pat_tag}"
    run_training_pipeline(config)

print("\nGrid complete.")